# Construindo uma Rede Neural

### Cenário
Uma empresa de e-commerce deseja estimar a **probabilidade de atraso de uma entrega**.

Cada pedido será representado por três características já normalizadas:

- `distancia`
- `quantidade_itens`
- `frete`

O alvo é:

\
y =
\begin{cases}
0 & \text{entrega no prazo}\\
1 & \text{entrega atrasada}
\end{cases}


### Objetivo

Ao longo desta atividade, você irá construir uma pequena rede neural para classificação binária só utilizando tensorflow.

Ao final, deverá compreender o fluxo:

> **Regra da aula:** não utilize `model.fit()`. Hoje queremos entender como a rede **produz uma predição**. O treinamento será estudado na próxima aula.

## Preparação

Execute a célula abaixo para importar as bibliotecas necessárias.

In [13]:
import tensorflow as tf
import numpy as np

print("TensorFlow:", tf.__version__)

TensorFlow: 2.20.0


# Tarefa 1: Representando os dados

Considere os seguintes pedidos:

| Pedido | Distância | Quantidade de itens | Frete | Atrasou? |
|---|---:|---:|---:|---:|
| 1 | 0.20 | 0.10 | 0.30 | 0 |
| 2 | 0.80 | 0.70 | 0.90 | 1 |
| 3 | 0.30 | 0.20 | 0.40 | 0 |
| 4 | 0.90 | 0.80 | 0.70 | 1 |
| 5 | 0.15 | 0.30 | 0.20 | 0 |
| 6 | 0.75 | 0.60 | 0.85 | 1 |

### Sua tarefa

Crie:

- um tensor `X` contendo as três features;
- um tensor `y` contendo o alvo.

Utilize `dtype=tf.float32`.

Depois, exiba os shapes de `X` e `y`.

### Antes de programar

Responda:

**Qual deve ser o shape de `X`?**

Resposta: (6, 3) 

**Qual deve ser o shape de `y`?**

Resposta: (6, )


In [14]:
X = tf.constant([
    [0.20, 0.10, 0.30],
    [0.80, 0.70, 0.90],
    [0.30, 0.20, 0.40],
    [0.90, 0.80, 0.70],
    [0.15, 0.30, 0.20],
    [0.75, 0.60, 0.85]
], dtype=tf.float32)

y = tf.constant(
    [0, 1, 0, 1, 0, 1],
    dtype=tf.float32
)

print(X.shape)
print(y.shape)


(6, 3)
(6,)


### Verificação conceitual

Complete:

Cada **linha** de `X` representa um pedido.

Cada **coluna** de `X` representa uma feature do pedido.

O valor `y = 1` significa que o pedido atrasou.


# Tarefa 2: O que acontece dentro de um neurônio?

Antes de construir a rede, vamos lembrar a operação fundamental de um neurônio:

$$
\boxed{z = \sum_i x_i w_i + b}
$$

Considere o primeiro pedido:

$$
x = [0.20,\ 0.10,\ 0.30]
$$

e os seguintes parâmetros:

$$
w = [0.5,\ -0.3,\ 0.8]
$$

$$
b = 0.1
$$

### Primeiro calcule manualmente

Complete a expressão:

z=(0.20×0.5)+(0.10×−0.3)+(0.30×0.8)+0.1

Resultado esperado por você:

z = 0.41

### Depois confirme utilizando TensorFlow.

In [15]:
w = tf.constant([0.5, -0.3, 0.8], dtype=tf.float32)
b = tf.constant(0.1, dtype=tf.float32)

x = X[0]

z = tf.reduce_sum(x * w) + b

print("z =", z.numpy())

z = 0.41


### Pergunta

O valor de \(z\) representa diretamente a probabilidade de atraso?

**( ) Sim**

**(X) Não**

Justifique:

z é o resultado da combinação linear das entradas, pesos e bias. Para obter uma probabilidade, ainda é necessário aplicar uma função de ativação apropriada, como a sigmoid na saída da rede.


# Tarefa 3: Função de ativação

Até agora, o neurônio calculou uma combinação das entradas:

$$
z = \sum_i x_iw_i + b
$$

Agora precisamos aplicar uma **função de ativação**.

Nesta tarefa, utilizaremos a **ReLU (Rectified Linear Unit)**:

$$
\boxed{\operatorname{ReLU}(z)=\max(0,z)}
$$

Isso significa que a ReLU compara o valor de $z$ com zero e retorna o **maior deles**:

- se $z < 0$, a saída será **0**;
- se $z > 0$, a saída será o **próprio $z$**.

### Antes de executar o código

Complete mentalmente a tabela:

| $z$ | $\operatorname{ReLU}(z)$ |
|---:|---:|
| -3 | 0 |
| -0.5 | 0 |
| 0 | 0 |
| 2 | 2 |
| 5 | 5 |

### Agora confirme utilizando TensorFlow

Aplique a função ReLU aos mesmos valores e compare o resultado com suas respostas.

In [16]:
valores = tf.constant([-3.0, -0.5, 0.0, 2.0, 5.0])

resultado = tf.nn.relu(valores)

print(resultado.numpy())

[0. 0. 0. 2. 5.]


### Pense antes de continuar

Por que uma rede neural precisa de funções de ativação **não lineares**?

____________________________________________________________________
Porque elas permitem que a rede neural aprenda relações complexas e não lineares entre as entradas e a saída. Sem elas, várias camadas lineares poderiam ser reduzidas a uma única transformação linear.
____________________________________________________________________


# Tarefa 4: Projetando a arquitetura

Agora vamos construir uma **MLP (Multilayer Perceptron)** para o nosso problema de classificação.

Cada pedido possui **3 características de entrada**:

- distância;
- quantidade de itens;
- valor do frete.

Nossa rede terá a seguinte arquitetura:

$$
\boxed{3 \rightarrow 4 \rightarrow 1}
$$

Podemos interpretá-la como:

$$
\text{3 entradas}
\rightarrow
\text{4 neurônios}
\rightarrow
\text{1 saída}
$$

A configuração será:

| Camada | Quantidade | Função de ativação |
|---|---:|---|
| Entrada | 3 valores | — |
| Camada escondida | 4 neurônios | ReLU |
| Saída | 1 neurônio | Sigmoid |


### Pense no fluxo

Complete:

$$
X
\rightarrow
\boxed{\text{Dense}(4)}
\rightarrow
\boxed{\ ?\ }
\rightarrow
\boxed{\text{Dense}(1)}
\rightarrow
\boxed{\ ?\ }
\rightarrow
\hat{y}
$$

Agora vamos transformar essa arquitetura em código utilizando TensorFlow.

### Agora construa a rede

Use:

- `tf.keras.Sequential`
- `tf.keras.layers.Input`
- `tf.keras.layers.Dense`

> **Dica:** a primeira camada `Dense` deve ter 4 neurônios e a segunda deve ter 1.


In [17]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(3,)),
    tf.keras.layers.Dense(4, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

# Tarefa 5: Quantos parâmetros existem?

Nossa rede possui:

$$
\boxed{3 \rightarrow 4 \rightarrow 1}
$$

## 1. Camada escondida

A camada escondida recebe **3 valores** e possui **4 neurônios**.

$$
n_{in}=3
$$

$$
n_{out}=4
$$

Pesos:

$$
3 \times 4 = 12
$$

Biases:

$$
4
$$

Total:

$$
12 + 4 = \boxed{16}
$$

## 2. Camada de saída

A camada de saída recebe os **4 neurônios anteriores** e possui **1 neurônio**.

$$
n_{in}=4
$$

$$
n_{out}=1
$$

Pesos:

$$
4 \times 1 = 4
$$

Bias:

$$
1
$$

Total:

$$
4 + 1 = \boxed{5}
$$

## 3. Total da rede

$$
16 + 5 = \boxed{21}
$$

Portanto, a rede possui **21 parâmetros treináveis**.


In [18]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_2 (Dense)                 │ (None, 4)              │            16 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │             5 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21 (84.00 B)

 Trainable params: 21 (84.00 B)

 Non-trainable params: 0 (0.00 B)

### Conferência

**Sim.**

O cálculo manual resulta em **21 parâmetros**, que deve coincidir com o valor apresentado pelo `model.summary()`.

# Tarefa 6: Forward Pass

A arquitetura da nossa rede está pronta. Agora vamos **passar os dados pela rede** e observar o que ela produz.

Esse processo é chamado de **forward pass** (propagação para frente).

Durante o forward pass, os dados percorrem a rede da entrada até a saída:

$$
X
\rightarrow
\boxed{\text{Dense}(4)}
\rightarrow
\boxed{\text{ReLU}}
\rightarrow
\boxed{\text{Dense}(1)}
\rightarrow
\boxed{\text{Sigmoid}}
\rightarrow
\hat{y}
$$

Em outras palavras:

1. a rede recebe as características de cada pedido;
2. a camada escondida calcula combinações usando **pesos e biases**;
3. a **ReLU** é aplicada;
4. a camada de saída realiza uma nova combinação;
5. a **sigmoid** transforma o resultado em um valor entre 0 e 1.

O resultado final será:

$$
\hat{y}
$$

onde $\hat{y}$ representa a **probabilidade estimada pelo modelo** de o pedido pertencer à classe positiva (`Atrasou = 1`).

---

## Sua tarefa

Passe o tensor `X` pelo modelo e armazene o resultado em:

```python
y_pred

In [19]:
y_pred = model(X)

print(y_pred.numpy())

[[0.5665891 ]
 [0.6662602 ]
 [0.5840252 ]
 [0.64746726]
 [0.49894023]
 [0.67371386]]


# Tarefa 7: Probabilidade não é classe

Após o forward pass, a saída da sigmoid está entre 0 e 1 e pode ser interpretada como a probabilidade estimada de atraso.

Para:

$$
\hat{y}=0.78
$$

### A. O que significa?

Significa que o modelo estima **78% de probabilidade de o pedido atrasar**, ou seja, de pertencer à classe positiva `1`.

### B. Threshold = 0.5

Como:

$$
0.78 \geq 0.5
$$

a previsão será:

$$
\boxed{\text{classe}=1}
$$

### C. Threshold = 0.85

Como:

$$
0.78 < 0.85
$$

a previsão será:

$$
\boxed{\text{classe}=0}
$$

### D. A rede mudou?

**Não.**

Alterar o threshold não modifica os pesos ou biases da rede. Apenas altera a regra utilizada para transformar a probabilidade em uma classe.

### Fluxo final

$$
X
\rightarrow
\text{Rede Neural}
\rightarrow
\hat{y}
\rightarrow
\text{Threshold}
\rightarrow
\text{Classe}
$$

A **rede neural termina ao produzir a probabilidade** `ŷ`. A **regra de decisão começa ao aplicar o threshold** para transformar essa probabilidade em uma classe.


## Aplicando um threshold

Agora transforme as probabilidades da sua rede em classes usando threshold `0.5`.

In [20]:
threshold = 0.5

y_class = ...

print("Probabilidades:")
print(y_pred.numpy())

print("\nClasses:")
print(...)

Probabilidades:
[[0.5665891 ]
 [0.6662602 ]
 [0.5840252 ]
 [0.64746726]
 [0.49894023]
 [0.67371386]]

Classes:
Ellipsis


# Tarefa 8: Comparando previsão e realidade

Agora exiba, para cada pedido:

- valor real;
- probabilidade prevista;
- classe prevista.

Complete o código.


In [24]:
for i in range(len(y)):
    real = ...
    probabilidade = ...
    classe = ...

    print(
        f"Pedido {i+1}: "
        f"real={real} | "
        f"probabilidade={probabilidade:.2f} | "
        f"classe={classe}"
    )

TypeError: unsupported format string passed to ellipsis.__format__

# Tarefa 9: Como medir o erro?

O valor verdadeiro é:

$$
y=1
$$

Portanto, quanto mais próxima de **1** estiver a probabilidade prevista, melhor será a previsão.

### A. Melhor previsão

**Modelo A**, com probabilidade `0.95`.

### B. Pior previsão

**Modelo C**, com probabilidade `0.05`.

### C. Maior penalização

**Modelo C**, porque atribuiu uma probabilidade muito baixa à classe correta e ainda fez uma previsão errada com alta confiança.

### D. Raciocínio

Como o valor real é `1`, a previsão `0.95` está muito próxima do valor correto. A previsão `0.60` ainda aponta para a classe correta, mas com menor confiança. Já `0.05` está muito distante de `1`, indicando uma previsão errada e confiante.

A função de Loss transforma essa diferença entre previsão e realidade em um valor numérico.


## Binary Cross-Entropy

Para problemas de **classificação binária**, uma função de Loss muito utilizada é a **Binary Cross-Entropy (BCE)**.

A ideia é simples:

> **Quanto maior a probabilidade atribuída à resposta correta, menor será a Loss.**

A fórmula é:

$$
\boxed{
L = -\left[
y\log(\hat{y}) +
(1-y)\log(1-\hat{y})
\right]
}
$$

onde:

- $y$ é o **valor verdadeiro** (`0` ou `1`);
- $\hat{y}$ é a **probabilidade prevista pelo modelo**;
- $L$ é a **Loss** da previsão.

---

### No nosso exemplo

Sabemos que:

$$
y=1
$$

Substituindo $y=1$ na fórmula:

$$
L = -\left[
1\log(\hat{y}) +
(1-1)\log(1-\hat{y})
\right]
$$

Como $(1-1)=0$:

$$
\boxed{L=-\log(\hat{y})}
$$

Portanto:

- $\hat{y}$ próximo de **1** → Loss pequena;
- $\hat{y}$ próximo de **0** → Loss grande.

Agora vamos verificar essa intuição utilizando `BinaryCrossentropy` do TensorFlow.

In [ ]:
loss_fn = tf.keras.losses.BinaryCrossentropy()

y_true = tf.constant([[1.0]])

predicoes = [
    tf.constant([[0.95]]),
    tf.constant([[0.60]]),
    tf.constant([[0.05]])
]

for predicao in predicoes:
    loss = ...
    print(
        "Predição:",
        float(predicao.numpy()[0, 0]),
        "| Loss:",
        ...
    )

Predição: 0.949999988079071 | Loss: Ellipsis
Predição: 0.6000000238418579 | Loss: Ellipsis
Predição: 0.05000000074505806 | Loss: Ellipsis


### Interpretação

Quanto **melhor** a previsão, geralmente **menor** será a loss.

Quanto mais **errada e confiante** a previsão, geralmente **maior** será a loss.

Para o exemplo `y=1`, os valores aproximados são:

| Predição | Loss aproximada |
|---:|---:|
| 0.95 | 0.051 |
| 0.60 | 0.511 |
| 0.05 | 2.996 |


# Tarefa 10: Loss da nossa rede

Agora utilize:

- os valores verdadeiros `y`;
- as probabilidades `y_pred`;

para calcular a Binary Cross-Entropy da nossa rede.

> **Importante:** utilize `y_pred`, e não as classes obtidas após o threshold.


In [22]:

loss_fn = tf.keras.losses.BinaryCrossentropy()

y_true = tf.reshape(y, (-1, 1))

loss = loss_fn(y_true, y_pred)

print("Loss:", loss.numpy())

Loss: 0.60665697


# Desafio final: Como a rede aprende?

Até agora, nossa rede consegue:

1. receber os dados de entrada;
2. realizar transformações;
3. produzir probabilidades;
4. comparar as previsões com os valores reais;
5. calcular a **Loss**.

O processo atual é:

$$
X
\rightarrow
\text{Rede Neural}
\rightarrow
\hat{y}
\rightarrow
\text{Loss}
$$

### 1. Como a rede poderia descobrir se um peso deve aumentar ou diminuir?

**Utilizando os gradientes, que indicam como a Loss varia em relação a cada parâmetro da rede.**

### 2. Como descobrir quanto esse peso deve mudar?

**O gradiente é combinado com uma taxa de aprendizado (`learning rate`), que define o tamanho da atualização do parâmetro.**

### 3. Como saber se as mudanças estão realmente melhorando o modelo?

**Observando se a Loss diminui após as atualizações. Se a Loss estiver diminuindo, isso indica que as alterações nos parâmetros estão melhorando as previsões.**

O fluxo de aprendizado será:

$$
\boxed{
\text{Loss}
\rightarrow
\text{Gradientes}
\rightarrow
\text{Backpropagation}
\rightarrow
\text{Gradient Descent}
\rightarrow
\text{Atualização dos parâmetros}
}
$$

> **Observação:** esta atividade não utiliza `model.fit()`. O objetivo é compreender o forward pass e a Loss antes de estudar o treinamento da rede.


---

# Checklist de aprendizagem

Antes de encerrar, marque o que você consegue explicar sem consultar o notebook:

- [ ] O que é um neurônio artificial.
- [ ] O papel de pesos e bias.
- [ ] O que representa \(z=Wx+b\).
- [ ] Por que utilizamos funções de ativação.
- [ ] Quando utilizar ReLU.
- [ ] Por que utilizamos sigmoid na saída deste problema.
- [ ] O que significa `Dense(4)`.
- [ ] Como calcular o número de parâmetros de uma camada Dense.
- [ ] O que é forward pass.
- [ ] Diferença entre probabilidade e classe.
- [ ] O papel do threshold.
- [ ] O que é uma função de loss.
- [ ] Por que a rede ainda não está aprendendo neste notebook.

## Próxima aula

**Como uma rede neural aprende?**
